# ÉPICA 1: Análisis de la Documentación

Herramienta de anonimización documental para ECOOO

## Objetivos
- HU-1.1: Inventario documental (clasificación, duplicados, escaneados)
- HU-1.2: Inventario de datos personales (campos identificables)
- HU-1.3: Clasificación por estrategia de anonimización

## Importes y configuración

In [1]:
import os
import json
import hashlib
import re
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Dict, List, Set, Tuple
import pandas as pd
import PyPDF2

# Configuración
DOCS_DIR = Path("/Users/usuario/Desktop/anonimizacion_ecooo/datos")
OUTPUT_DIR = Path("/Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Directorio de documentos: {DOCS_DIR}")
print(f"Directorio de salida: {OUTPUT_DIR}")

Directorio de documentos: /Users/usuario/Desktop/anonimizacion_ecooo/datos
Directorio de salida: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output


## Arquitectura de Datos

In [2]:
@dataclass
class DocumentInfo:
    """Información de un documento"""
    filename: str
    filepath: str
    extension: str
    size_bytes: int
    file_hash: str
    is_scanned: bool = False
    personal_data: Dict[str, List[str]] = None
    strategy: str = None

@dataclass
class PersonalDataPattern:
    """Patrón para detectar datos personales"""
    name: str
    regex: str
    category: str  # 'identifier', 'contact', 'location', 'financial'
    confidence: float  # 0.0-1.0

# Patrones de datos personales
PII_PATTERNS = [
    PersonalDataPattern(
        name="DNI/NIE",
        regex=r'(?:DNI|NIE|CIF)[-\s]?(?:[0-9]{7,8}|[0-9X]{7,8}[A-Z])',
        category="identifier",
        confidence=0.95
    ),
    PersonalDataPattern(
        name="Email",
        regex=r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        category="contact",
        confidence=0.90
    ),
    PersonalDataPattern(
        name="Teléfono",
        regex=r'(?:\+\d{1,3})?[-\.\s]?(?:\(?0?\d{2,4}\)?)?[-\.\s]?\d{3,4}[-\.\s]?\d{3,4}',
        category="contact",
        confidence=0.85
    ),
    PersonalDataPattern(
        name="CUPS",
        regex=r'\b[A-Z]{2}\d{22}\b',
        category="location",
        confidence=0.98
    ),
    PersonalDataPattern(
        name="Coordenadas",
        regex=r'\b-?\d{1,3}\.\d+,\s*-?\d{1,3}\.\d+\b',
        category="location",
        confidence=0.90
    ),
    PersonalDataPattern(
        name="IBAN",
        regex=r'(?:ES\d{2}\s?\d{4}\s?\d{4}\s?\d{1,2}\s?[A-Z0-9]{0,30})',
        category="financial",
        confidence=0.95
    ),
]

print(f"Patrones definidos: {len(PII_PATTERNS)}")
for pattern in PII_PATTERNS:
    print(f"  - {pattern.name}: {pattern.category} (confianza: {pattern.confidence})")

Patrones definidos: 6
  - DNI/NIE: identifier (confianza: 0.95)
  - Email: contact (confianza: 0.9)
  - Teléfono: contact (confianza: 0.85)
  - CUPS: location (confianza: 0.98)
  - Coordenadas: location (confianza: 0.9)
  - IBAN: financial (confianza: 0.95)


## HU-1.1: Inventario Documental

In [3]:
class DocumentInventory:
    """Gestor de inventario documental"""
    
    def __init__(self, docs_dir: Path):
        self.docs_dir = docs_dir
        self.documents: List[DocumentInfo] = []
        self.file_types: Dict[str, int] = defaultdict(int)
        self.file_hashes: Dict[str, List[str]] = defaultdict(list)
        self.duplicates: Dict[str, List[str]] = {}
    
    def scan_directory(self) -> None:
        """Escanea el directorio y construye el inventario"""
        print(f"Escaneando {self.docs_dir}...")
        
        for root, dirs, files in os.walk(self.docs_dir):
            for file in files:
                if file.startswith('.'):
                    continue
                    
                filepath = Path(root) / file
                ext = filepath.suffix.lower()
                
                # Contar tipo
                self.file_types[ext] += 1
                
                # Calcular hash
                try:
                    with open(filepath, 'rb') as f:
                        file_hash = hashlib.md5(f.read()).hexdigest()
                    
                    rel_path = str(filepath.relative_to(self.docs_dir))
                    self.file_hashes[file_hash].append(rel_path)
                    
                    doc_info = DocumentInfo(
                        filename=file,
                        filepath=rel_path,
                        extension=ext,
                        size_bytes=filepath.stat().st_size,
                        file_hash=file_hash
                    )
                    self.documents.append(doc_info)
                except Exception as e:
                    print(f"  ⚠️ Error procesando {file}: {e}")
        
        # Detectar duplicados
        self.duplicates = {h: files for h, files in self.file_hashes.items() if len(files) > 1}
        print(f"✅ Escaneo completado: {len(self.documents)} documentos")
    
    def detect_scanned_pdfs(self) -> None:
        """Detecta PDFs escaneados (sin texto extraíble)"""
        print("Detectando PDFs escaneados...")
        
        for doc in self.documents:
            if doc.extension == '.pdf':
                try:
                    filepath = self.docs_dir / doc.filepath
                    with open(filepath, 'rb') as f:
                        reader = PyPDF2.PdfReader(f)
                        
                        # Extraer texto de primeras páginas
                        text_chars = 0
                        for page in reader.pages[:3]:
                            try:
                                text_chars += len(page.extract_text())
                            except:
                                pass
                        
                        # Si pocas caracteres, es escaneado
                        doc.is_scanned = text_chars < 100
                except:
                    pass
    
    def generate_report(self) -> Dict:
        """Genera reporte del inventario"""
        scanned_count = sum(1 for d in self.documents if d.is_scanned)
        
        report = {
            "total_documents": len(self.documents),
            "by_type": dict(self.file_types),
            "scanned_pdfs": scanned_count,
            "duplicates": len(self.duplicates),
            "duplicate_documents": len([f for files in self.duplicates.values() for f in files])
        }
        return report

# Ejecutar HU-1.1
inventory = DocumentInventory(DOCS_DIR)
inventory.scan_directory()
inventory.detect_scanned_pdfs()

report = inventory.generate_report()
print(f"\n📊 REPORTE HU-1.1:")
print(json.dumps(report, indent=2))

Escaneando /Users/usuario/Desktop/anonimizacion_ecooo/datos...


✅ Escaneo completado: 3669 documentos
Detectando PDFs escaneados...



📊 REPORTE HU-1.1:
{
  "total_documents": 3669,
  "by_type": {
    ".pdf": 3457,
    ".docx": 190,
    ".xlsx": 21,
    ".zip": 1
  },
  "scanned_pdfs": 22,
  "duplicates": 655,
  "duplicate_documents": 1570
}


## HU-1.2: Inventario de Datos Personales

In [4]:
class PersonalDataDetector:
    """Detecta datos personales en documentos"""
    
    def __init__(self, patterns: List[PersonalDataPattern]):
        self.patterns = patterns
        self.compiled_patterns = {p.name: (re.compile(p.regex), p) for p in patterns}
    
    def extract_text_from_pdf(self, filepath: Path) -> str:
        """Extrae texto de un PDF"""
        try:
            with open(filepath, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                text = ""
                for page in reader.pages[:5]:  # Primeras 5 páginas
                    text += page.extract_text()
                return text
        except:
            return ""
    
    def detect_pii_in_text(self, text: str) -> Dict[str, List[str]]:
        """Detecta PII en un texto"""
        findings = {}
        
        for pattern_name, (compiled_re, pattern) in self.compiled_patterns.items():
            matches = compiled_re.findall(text)
            if matches:
                findings[pattern_name] = matches
        
        return findings
    
    def analyze_sample(self, documents: List[DocumentInfo], sample_size: int = 50) -> Dict:
        """Analiza una muestra de documentos"""
        print(f"Analizando muestra de {sample_size} documentos...")
        
        findings_summary = defaultdict(int)
        sample_files = defaultdict(list)
        
        pdf_docs = [d for d in documents if d.extension == '.pdf'][:sample_size]
        
        for doc in pdf_docs:
            filepath = DOCS_DIR / doc.filepath
            text = self.extract_text_from_pdf(filepath)
            
            findings = self.detect_pii_in_text(text)
            doc.personal_data = findings
            
            for field, matches in findings.items():
                findings_summary[field] += len(matches)
                if len(sample_files[field]) < 3:
                    sample_files[field].append(doc.filename)
        
        return {
            "sample_size": len(pdf_docs),
            "findings": dict(findings_summary),
            "sample_files": dict(sample_files)
        }

# Ejecutar HU-1.2
detector = PersonalDataDetector(PII_PATTERNS)
pii_report = detector.analyze_sample(inventory.documents, sample_size=50)

print(f"\n📋 REPORTE HU-1.2 (Datos Personales):")
print(json.dumps(pii_report, indent=2))

Analizando muestra de 50 documentos...



📋 REPORTE HU-1.2 (Datos Personales):
{
  "sample_size": 50,
  "findings": {
    "Email": 77,
    "Tel\u00e9fono": 382,
    "IBAN": 13,
    "DNI/NIE": 3,
    "CUPS": 2
  },
  "sample_files": {
    "Email": [
      "22e478c8-8649-4e1d-afd4-25607d98a5b1ALRE016 just_registro_09-12-24.pdf",
      "2f038919-c0fa-4bef-8773-881e11de96c3IMS333_CEE_V1.pdf",
      "456cc0b8-864f-4fe3-b6ca-66057419901dDelta B6 3A_CEE_V1.pdf"
    ],
    "Tel\u00e9fono": [
      "22e478c8-8649-4e1d-afd4-25607d98a5b1ALRE016 just_registro_09-12-24.pdf",
      "2f038919-c0fa-4bef-8773-881e11de96c3IMS333_CEE_V1.pdf",
      "456cc0b8-864f-4fe3-b6ca-66057419901dDelta B6 3A_CEE_V1.pdf"
    ],
    "IBAN": [
      "7d917a52-ddb7-40ce-b4e5-33ad3a2170951. LRE341 CIE fd.pdf",
      "4bdc6909-bb69-42e9-9ff1-d790f162f34dANEXO V RELLENO C287.pdf",
      "419d029f-697e-47c4-9e26-2b5d394fc710ANEXO V RELLENO-2.pdf"
    ],
    "DNI/NIE": [
      "4e445bcd-581b-4773-ba90-488604312b4e8.2. Solicitud Subvenci\u00f3n.pdf",
      "c44b0699

## HU-1.3: Clasificación por Estrategia

In [5]:
class AnonymizationStrategyClassifier:
    """Clasifica documentos por estrategia de anonimización"""
    
    STRATEGIES = {
        ".pdf": "PyMuPDF + Redacción",
        ".pdf_scanned": "OCR + PyMuPDF + Redacción",
        ".docx": "python-docx + Búsqueda-Reemplazo",
        ".xlsx": "openpyxl + Búsqueda-Reemplazo",
    }
    
    def classify(self, documents: List[DocumentInfo]) -> Dict:
        """Clasifica todos los documentos"""
        strategies = defaultdict(list)
        
        for doc in documents:
            if doc.extension == ".pdf":
                strategy = "PyMuPDF + OCR" if doc.is_scanned else "PyMuPDF"
                doc.strategy = strategy
            elif doc.extension == ".docx":
                doc.strategy = "python-docx"
            elif doc.extension == ".xlsx":
                doc.strategy = "openpyxl"
            else:
                doc.strategy = "Manual"
            
            strategies[doc.strategy].append(doc.filename)
        
        return dict(strategies)
    
    def generate_strategy_report(self, strategies: Dict) -> None:
        """Genera reporte de estrategias"""
        print("\n📄 CLASIFICACIÓN POR ESTRATEGIA:")
        for strategy, files in strategies.items():
            print(f"\n  {strategy}: {len(files)} documentos")

# Ejecutar HU-1.3
classifier = AnonymizationStrategyClassifier()
strategies = classifier.classify(inventory.documents)
classifier.generate_strategy_report(strategies)


📄 CLASIFICACIÓN POR ESTRATEGIA:

  PyMuPDF: 3435 documentos

  python-docx: 190 documentos

  openpyxl: 21 documentos

  PyMuPDF + OCR: 22 documentos

  Manual: 1 documentos


## Exportar Resultados

In [6]:
# Exportar inventario como CSV
docs_df = pd.DataFrame([
    {
        "filename": d.filename,
        "extension": d.extension,
        "size_mb": d.size_bytes / (1024*1024),
        "is_scanned": d.is_scanned,
        "strategy": d.strategy,
        "pii_fields": len(d.personal_data) if d.personal_data else 0
    }
    for d in inventory.documents
])

csv_path = OUTPUT_DIR / "inventario_documentos.csv"
docs_df.to_csv(csv_path, index=False)
print(f"✅ Inventario exportado a: {csv_path}")

# Exportar reporte JSON
full_report = {
    "epic": "1",
    "hu_1_1": report,
    "hu_1_2": pii_report,
    "hu_1_3": {
        "strategies": {k: len(v) for k, v in strategies.items()}
    }
}

json_path = OUTPUT_DIR / "reporte_epica1.json"
with open(json_path, 'w') as f:
    json.dump(full_report, f, indent=2)
print(f"✅ Reporte exportado a: {json_path}")

print(f"\n✅ ÉPICA 1 COMPLETADA")

✅ Inventario exportado a: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output/inventario_documentos.csv
✅ Reporte exportado a: /Users/usuario/Desktop/anonimizacion_ecooo/notebooks/output/reporte_epica1.json

✅ ÉPICA 1 COMPLETADA
